In [1]:
# ============================================================
# CELL 1: Import pipeline from src/
# ============================================================
import sys
import os
sys.path.append('../src')

from url_pipeline import check_url, PHISHTANK_DB
import pandas as pd
import time

print("PSUDPS — Phishing Short URL Detection System")
print("=" * 50)
print("Pipeline loaded successfully!")
print(f"PhishTank DB: {'Available' if os.path.exists(PHISHTANK_DB) else 'Unavailable'}")

PSUDPS — Phishing Short URL Detection System
Pipeline loaded successfully!
PhishTank DB: Available


In [2]:
# ============================================================
# CELL 2: Test 10 URLs — mix of safe, suspicious, phishing
# ============================================================

test_urls = [
    # Known safe sites via short URLs
    "https://tinyurl.com/wikipedia-en",
    "https://bit.ly/3google",
    "https://tinyurl.com/python-docs",

    # Direct safe URLs
    "https://www.google.com",
    "https://www.github.com",

    # Suspicious looking URLs
    "https://cutt.ly/github",
    "https://rb.gy/stackoverflow",

    # Known phishing redirect abuses
    "https://www.google.com/url?q=https://evil-site.com",
    "https://docs.google.com/fake-login-page",
    "https://sites.google.com/view/fake-bank",
]

results_log = []

print("TESTING 10 URLs THROUGH PSUDPS PIPELINE")
print("=" * 60)

for i, url in enumerate(test_urls, 1):
    print(f"\n[{i}/10] Testing: {url}")
    result = check_url(url, db_path=PHISHTANK_DB, verbose=False)

    results_log.append({
        'No'           : i,
        'Input URL'    : result['input_url'],
        'Expanded URL' : str(result['expanded_url'])[:50] + '...'
                        if result['expanded_url'] and
                        len(str(result['expanded_url'])) > 50
                        else result['expanded_url'],
        'PhishTank'    : result['phishtank_result'],
        'ML Prob'      : f"{result['ml_result']['phishing_prob']}%"
                        if result.get('ml_result') else 'N/A',
        'Verdict'      : result['final_verdict'],
        'Reason'       : result['verdict_reason']
    })

    verdict = result['final_verdict']
    icon    = '🚨' if verdict == 'PHISHING' else (
              '⚠️ ' if verdict == 'SUSPICIOUS' else '✅')
    print(f"  {icon} {verdict} — {result['verdict_reason']}")
    time.sleep(0.5)

print("\n\nFINAL RESULTS TABLE")
print("=" * 60)
display(pd.DataFrame(results_log))

TESTING 10 URLs THROUGH PSUDPS PIPELINE

[1/10] Testing: https://tinyurl.com/wikipedia-en
  🚨 PHISHING — Found in PhishTank blacklist

[2/10] Testing: https://bit.ly/3google
  ⚠️  SUSPICIOUS — ML flagged but low confidence (55.8%) — monitor

[3/10] Testing: https://tinyurl.com/python-docs
  🚨 PHISHING — ML confidence 77.2% above 70% threshold

[4/10] Testing: https://www.google.com
  ⚠️  SUSPICIOUS — ML flagged but low confidence (55.8%) — monitor

[5/10] Testing: https://www.github.com
  ⚠️  SUSPICIOUS — ML flagged but low confidence (55.8%) — monitor

[6/10] Testing: https://cutt.ly/github
  ⚠️  SUSPICIOUS — ML flagged but low confidence (53.7%) — monitor

[7/10] Testing: https://rb.gy/stackoverflow
  🚨 PHISHING — ML confidence 79.6% above 70% threshold

[8/10] Testing: https://www.google.com/url?q=https://evil-site.com
  🚨 PHISHING — Flagged by BOTH PhishTank and ML model

[9/10] Testing: https://docs.google.com/fake-login-page
  🚨 PHISHING — Found in PhishTank blacklist

[10/10] Te

,No,Input URL,Expanded URL,PhishTank,ML Prob,Verdict,Reason
0,1,https://tinyurl.com/wikipedia-en,https://tinyurl.com/preview/wikipedia-en,PHISHING,37.0%,PHISHING,Found in PhishTank blacklist
1,2,https://bit.ly/3google,https://searchengineland.com/meet-the-new-goog...,SAFE,55.8%,SUSPICIOUS,ML flagged but low confidence (55.8%) — monitor
2,3,https://tinyurl.com/python-docs,https://cloud4greentechnology-my.sharepoint.co...,SAFE,77.2%,PHISHING,ML confidence 77.2% above 70% threshold
3,4,https://www.google.com,https://www.google.com/,SAFE,55.8%,SUSPICIOUS,ML flagged but low confidence (55.8%) — monitor
4,5,https://www.github.com,https://github.com/,SAFE,55.8%,SUSPICIOUS,ML flagged but low confidence (55.8%) — monitor
5,6,https://cutt.ly/github,https://bayuangora.github.io,SAFE,53.7%,SUSPICIOUS,ML flagged but low confidence (53.7%) — monitor
6,7,https://rb.gy/stackoverflow,https://free-url-shortener.rb.gy/,SAFE,79.6%,PHISHING,ML confidence 79.6% above 70% threshold
7,8,https://www.google.com/url?q=https://evil-site...,https://www.google.com/url?q=https://evil-site...,PHISHING,72.1%,PHISHING,Flagged by BOTH PhishTank and ML model
8,9,https://docs.google.com/fake-login-page,https://docs.google.com/fake-login-page,PHISHING,53.7%,PHISHING,Found in PhishTank blacklist
9,10,https://sites.google.com/view/fake-bank,https://sites.google.com/view/fake-bank,PHISHING,53.7%,PHISHING,Found in PhishTank blacklist


In [3]:
# ============================================================
# CELL 3: Edge case handling tests
# ============================================================

print("EDGE CASE TESTS")
print("=" * 50)

edge_cases = [
    ("not-a-url-at-all",          "Invalid format"),
    ("",                           "Empty string"),
    ("http://",                    "Incomplete URL"),
    ("https://expired-short.ly",   "Possibly expired"),
]

for url, description in edge_cases:
    result = check_url(url, db_path=PHISHTANK_DB, verbose=False)
    print(f"\nTest  : {description}")
    print(f"Input : '{url}'")
    print(f"Result: {result['final_verdict']}")
    if result.get('error'):
        print(f"Error : {result['error']}")

EDGE CASE TESTS

Test  : Invalid format
Input : 'not-a-url-at-all'
Result: INVALID URL
Error : URL must start with http:// or https://

Test  : Empty string
Input : ''
Result: INVALID URL
Error : Empty or invalid input

Test  : Incomplete URL
Input : 'http://'
Result: INVALID URL
Error : URL too short to be valid

Test  : Possibly expired
Input : 'https://expired-short.ly'
Result: PHISHING


In [4]:
# ============================================================
# CELL 4: Full verbose output demo — for presentation
# Shows every step of the pipeline clearly
# ============================================================

demo_url = "https://tinyurl.com/wikipedia-en"
result   = check_url(demo_url, db_path=PHISHTANK_DB, verbose=True)


PSUDPS — Phishing Short URL Detection System
Input : https://tinyurl.com/wikipedia-en
------------------------------------------------------------

Step 1: Unshortening URL...
  Original : https://tinyurl.com/wikipedia-en
  Expanded : https://tinyurl.com/preview/wikipedia-en
  Redirects: 1

Step 2: Checking PhishTank blacklist...
  FOUND IN BLACKLIST

Step 3: Running ML model...
  Phishing probability : 37.0%
  Legitimate probability: 63.0%
  Confidence           : 63.0%

VERDICT: PHISHING — BLOCKED
Reason : Found in PhishTank blacklist


In [5]:
# ============================================================
# CELL 5: Summary statistics of all 10 tests
# ============================================================

results_df = pd.DataFrame(results_log)

verdict_counts = results_df['Verdict'].value_counts()

print("RESULTS SUMMARY")
print("=" * 40)
print(f"Total URLs tested : {len(results_df)}")
print()
for verdict, count in verdict_counts.items():
    icon = '🚨' if verdict == 'PHISHING' else (
           '⚠️ ' if verdict == 'SUSPICIOUS' else '✅')
    print(f"  {icon} {verdict:<12}: {count}")

print()
print(f"Detection rate    : "
      f"{(verdict_counts.get('PHISHING', 0) / len(results_df) * 100):.0f}%")
print()
print("System is ready for Streamlit UI in Phase 6!")

RESULTS SUMMARY
Total URLs tested : 10

  🚨 PHISHING    : 6
  ⚠️  SUSPICIOUS  : 4

Detection rate    : 60%

System is ready for Streamlit UI in Phase 6!
